# Recognizing Faces in the Wild - Kinship Verification

This notebook is designed for Kaggle Notebooks. It assumes the FIW competition dataset is mounted under `/kaggle/input/recognizing-faces-in-the-wild/` and writes generated artifacts under `/kaggle/working/`.

This repository also includes `main.ipynb`, a local/full workflow that downloads Kaggle data, trains and evaluates the model, and generates a Kaggle submission file. This Kaggle version focuses on the mounted-data training and held-out evaluation workflow.

The model uses a Siamese CNN with a pretrained `facenet_pytorch.InceptionResnetV1(pretrained='vggface2')` backbone to predict kinship similarity from facial image pairs.

## Runtime assumptions

- The FIW competition dataset is attached as a Kaggle input at `/kaggle/input/recognizing-faces-in-the-wild/`.
- `/kaggle/working/` is writable and used for extracted images, saved model weights, and plots.
- Internet access or cached packages are available for `facenet_pytorch` and pretrained weights.
- GPU acceleration is recommended; runtime varies by Kaggle GPU type and package/download cache state.

## Data-use note

Before using the FIW competition data, review the Kaggle competition rules, especially the data-use requirements. Work using FIW data should cite the official FIW/RFIW papers listed in the competition documentation.


## Setup: Install dependencies

In [ ]:
!pip install facenet_pytorch -q
print('Dependencies installed.')

## Step 1: Load competition data from Kaggle input

In [ ]:
import os
import zipfile
import pandas
from collections import defaultdict
import glob

# Kaggle dataset path
dataset_path = '/kaggle/input/recognizing-faces-in-the-wild'

# Check structure
print('Dataset contents:')
for item in os.listdir(dataset_path):
    print(f'  {item}')

# Extract paths
train_zip = os.path.join(dataset_path, 'train-faces.zip')
test_zip = os.path.join(dataset_path, 'test-faces.zip')
relations_csv = os.path.join(dataset_path, 'train_relationships.csv')

# Use Kaggle's working directory for extraction
work_dir = '/kaggle/working'
train_faces_dir = os.path.join(work_dir, 'train-faces')
test_faces_dir = os.path.join(work_dir, 'test-faces')

# Extract training faces
if not os.path.exists(train_faces_dir):
    print('Extracting training faces...')
    with zipfile.ZipFile(train_zip, 'r') as z:
        z.extractall(work_dir)
    os.rename(os.path.join(work_dir, 'train_faces'), train_faces_dir)

# Extract test faces
if not os.path.exists(test_faces_dir):
    print('Extracting test faces...')
    with zipfile.ZipFile(test_zip, 'r') as z:
        z.extractall(work_dir)
    os.rename(os.path.join(work_dir, 'test_faces'), test_faces_dir)

print(f'Training faces: {len(os.listdir(train_faces_dir))} families')
print(f'Test faces: {len(os.listdir(test_faces_dir))} images')
print('Step 1 complete: Data loaded.')

## Step 2: Load and clean the labeled dataset

The `train_relationships.csv` file contains labeled kinship pairs in `family/member` format. Rows are removed when either member does not have corresponding image data in the extracted training faces directory.


In [ ]:
# Load relations
print(f'Loading {relations_csv}')
relations_df = pandas.read_csv(relations_csv, delimiter=',', header='infer')

# Create a dictionary to lookup image files for each member
family_dict = defaultdict(list)
for family in glob.glob(os.path.join(train_faces_dir, '*')):
    for member in glob.glob(os.path.join(family, '*')):
        for image_path in glob.glob(os.path.join(member, '*')):
            member_id = os.path.basename(member)
            image_file = os.path.basename(image_path)
            family_dict[member_id].append(image_file)

print(f'Images found for {len(family_dict)} members')

# Remove entries which do not exist in the training set
print(f'Original relations: {len(relations_df)} pairs')
fam_keys = family_dict.keys()
missing_relations_list = []

for index, row in relations_df.iterrows():
    split1 = row.p1.split('/')
    split2 = row.p2.split('/')
    p1fam, p1member = split1[0], split1[1]
    p2fam, p2member = split2[0], split2[1]
    
    if (p1fam not in fam_keys or p2fam not in fam_keys or 
        p1member not in family_dict[p1fam] or p2member not in family_dict[p2fam]):
        missing_relations_list.append(index)
        continue
    
    images1 = os.listdir(os.path.join(train_faces_dir, p1fam, p1member))
    images2 = os.listdir(os.path.join(train_faces_dir, p2fam, p2member))
    if len(images1) == 0 or len(images2) == 0:
        missing_relations_list.append(index)

if missing_relations_list:
    relations_df = relations_df.drop(missing_relations_list)
    print(f'Removed {len(missing_relations_list)} pairs with missing data')

print(f'Final relations: {len(relations_df)} pairs')
print('Step 2 complete: Data cleaned.')

## Step 3: Split by family and generate balanced pairs

Pairs are split by family so that members of the same family do not appear across training, validation, and held-out test splits. This reduces leakage from shared family identity across splits.

Positive pairs come from labeled kinship relationships. Negative pairs are sampled from members without a listed relationship and are treated as presumed unrelated for training balance; they are not independently verified non-kin pairs.


In [ ]:
import itertools
import random
import numpy
import torch

# Set seeds for reproducibility
random.seed(42)
numpy.random.seed(42)
torch.manual_seed(42)

# Extract unique family IDs and split 70/15/15
all_families = sorted(set(row.p1.split('/')[0] for _, row in relations_df.iterrows()))
random.shuffle(all_families)
n = len(all_families)
train_cutoff = int(0.70 * n)
val_cutoff = int(0.85 * n)
train_families = set(all_families[:train_cutoff])
val_families = set(all_families[train_cutoff:val_cutoff])
test_families = set(all_families[val_cutoff:])
print(f'Family split: {len(train_families)} train, {len(val_families)} val, {len(test_families)} test')

# Build member image lookup
members = sorted(set(m for row in relations_df.values for m in row))
member_images = dict()
for member in members:
    member_path = os.path.join(train_faces_dir, member)
    if os.path.exists(member_path):
        image_files = os.listdir(member_path)
        if len(image_files) > 0:
            member_images[member] = [os.path.join(member, img) for img in image_files]

def generate_pairs(relations_subset, family_set):
    """Generate balanced positive and negative pairs."""
    positives = []
    for _, row in relations_subset.iterrows():
        p1_images = member_images.get(row.p1, [])
        p2_images = member_images.get(row.p2, [])
        for img1, img2 in itertools.product(p1_images, p2_images):
            positives.append([img1, img2, 1.0])

    split_members = [m for m in members if m.split('/')[0] in family_set and m in member_images]
    negatives = []
    while len(negatives) < len(positives):
        p1 = random.choice(split_members)
        p2 = random.choice(split_members)
        if p1 == p2:
            continue
        p1_images = member_images[p1]
        p2_images = member_images[p2]
        for img1, img2 in itertools.product(p1_images, p2_images):
            negatives.append([img1, img2, 0.0])
            if len(negatives) >= len(positives):
                break

    data = positives + negatives[:len(positives)]
    random.shuffle(data)
    return data, len(positives), len(negatives[:len(positives)])

# Split relations
train_relations = relations_df[relations_df['p1'].apply(lambda x: x.split('/')[0] in train_families)]
val_relations = relations_df[relations_df['p1'].apply(lambda x: x.split('/')[0] in val_families)]
test_relations = relations_df[relations_df['p1'].apply(lambda x: x.split('/')[0] in test_families)]

training_data, train_pos, train_neg = generate_pairs(train_relations, train_families)
val_data, val_pos, val_neg = generate_pairs(val_relations, val_families)
testing_data, test_pos, test_neg = generate_pairs(test_relations, test_families)

print(f'Training:   {len(training_data)} pairs ({train_pos} pos + {train_neg} neg)')
print(f'Validation: {len(val_data)} pairs ({val_pos} pos + {val_neg} neg)')
print(f'Testing:    {len(testing_data)} pairs ({test_pos} pos + {test_neg} neg)')
print('Step 3 complete: Data split.')

## Step 4: Create DataLoaders

The training dataset uses image augmentation, while validation and held-out test datasets use deterministic preprocessing. Images are resized and normalized for the pretrained face encoder.


In [ ]:
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image

# Transforms with augmentation
train_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class ImagePairDataset(Dataset):
    def __init__(self, data, transform, image_dir):
        self.image_pairs = [sublist[:-1] for sublist in data]
        self.labels = torch.tensor([sublist[-1] for sublist in data], dtype=torch.float32)
        self.transform = transform
        self.image_dir = image_dir

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img0_path = os.path.join(self.image_dir, self.image_pairs[idx][0])
        img1_path = os.path.join(self.image_dir, self.image_pairs[idx][1])
        img0 = Image.open(img0_path)
        img1 = Image.open(img1_path)
        img0 = self.transform(img0)
        img1 = self.transform(img1)
        return img0, img1, self.labels[idx]

batch_size = 64

train_dataset = ImagePairDataset(training_data, train_transform, train_faces_dir)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = ImagePairDataset(val_data, eval_transform, train_faces_dir)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

test_dataset = ImagePairDataset(testing_data, eval_transform, train_faces_dir)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test:  {len(test_dataset)} samples, {len(test_loader)} batches')
print('Step 4 complete: DataLoaders created.')

## Step 5: Build the model and optimizer

The model uses a Siamese architecture: the same pretrained `InceptionResnetV1` face encoder processes both images, producing embeddings in a shared space. Contrastive loss pulls related pairs closer together and pushes presumed unrelated pairs apart. During evaluation, lower embedding distance means stronger predicted kinship similarity.


In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from facenet_pytorch import InceptionResnetV1

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        backbone = InceptionResnetV1(pretrained='vggface2')
        
        for param in backbone.parameters():
            param.requires_grad = False
        
        for name, param in backbone.named_parameters():
            if name.startswith('repeat_3.') or name.startswith('block8.'):
                param.requires_grad = True
        
        self.backbone = nn.Sequential(
            backbone.conv2d_1a, backbone.conv2d_2a, backbone.conv2d_2b, backbone.maxpool_3a,
            backbone.conv2d_3b, backbone.conv2d_4a, backbone.conv2d_4b, backbone.repeat_1,
            backbone.mixed_6a, backbone.repeat_2, backbone.mixed_7a, backbone.repeat_3,
            backbone.block8, backbone.avgpool_1a, nn.Flatten(), backbone.dropout,
            backbone.last_linear, backbone.last_bn,
        )
        self.fc1 = nn.Linear(512, 128)

    def forward_once(self, x):
        output = self.backbone(x)
        output = self.fc1(output)
        return output

    def forward(self, input1, input2):
        return self.forward_once(input1), self.forward_once(input2)

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
        euclidean_distance = F.pairwise_distance(output1, output2, keepdim=True)
        loss_contrastive = torch.mean((label) * torch.pow(euclidean_distance, 2) +
                                      (1-label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))
        return loss_contrastive

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SiameseNetwork().to(device)
criterion = ContrastiveLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

print(f'Model initialized on device: {device}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

## Step 6: Train the model

In [ ]:
def train(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for data1, data2, label in dataloader:
        data1, data2, label = data1.to(device), data2.to(device), label.to(device)
        optimizer.zero_grad()
        output1, output2 = model(data1, data2)
        loss = criterion(output1, output2, label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data1.size(0)
    return running_loss / len(dataloader.dataset)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for data1, data2, label in dataloader:
            data1, data2, label = data1.to(device), data2.to(device), label.to(device)
            output1, output2 = model(data1, data2)
            loss = criterion(output1, output2, label)
            running_loss += loss.item() * data1.size(0)
    return running_loss / len(dataloader.dataset)

num_epochs = 10
print(f'Training for {num_epochs} epochs...\n')

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:2d}/{num_epochs} — Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('\nTraining complete!')
torch.save(model.state_dict(), os.path.join(work_dir, 'model.pth'))

## Step 7: Training-set sanity check

This check runs on the training loader and verifies the expected direction of the learned distances: related pairs should be closer than unrelated pairs.


In [ ]:
# Verify related pairs are closer than unrelated
model.eval()
pos_distances = []
neg_distances = []

with torch.no_grad():
    for data1, data2, label in train_loader:
        data1, data2, label = data1.to(device), data2.to(device), label.to(device)
        output1, output2 = model(data1, data2)
        distances = F.pairwise_distance(output1, output2)
        for d, l in zip(distances, label):
            if l.item() == 1.0:
                pos_distances.append(d.item())
            else:
                neg_distances.append(d.item())

mean_pos = sum(pos_distances) / len(pos_distances)
mean_neg = sum(neg_distances) / len(neg_distances)

print(f'Mean distance (related pairs):   {mean_pos:.4f}')
print(f'Mean distance (unrelated pairs): {mean_neg:.4f}')
if mean_pos < mean_neg:
    print('✓ PASS: Related pairs are closer than unrelated pairs.')
else:
    print('✗ FAIL: Related pairs are NOT closer.')

## Step 8: Evaluate on the held-out labeled test split

This evaluates the model on the internal held-out labeled test split created from `train_relationships.csv`, not on the unlabeled Kaggle competition `test-faces` submission set.

AUC is threshold-independent. The threshold selected below is chosen on this same held-out split for inspection, so the derived accuracy, precision, and recall should be treated as exploratory diagnostics rather than separately validated deployment metrics.

Lower embedding distance means stronger predicted kinship similarity. The score used for metrics is an inverted normalized distance, not a calibrated probability.


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score
import numpy as np
import matplotlib.pyplot as plt

def test(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_distances = []
    all_labels = []
    with torch.no_grad():
        for data1, data2, label in dataloader:
            data1, data2, label = data1.to(device), data2.to(device), label.to(device)
            output1, output2 = model(data1, data2)
            loss = criterion(output1, output2, label)
            running_loss += loss.item() * data1.size(0)
            distances = F.pairwise_distance(output1, output2)
            all_distances.extend(distances.cpu().numpy())
            all_labels.extend(label.cpu().numpy())
    epoch_loss = running_loss / len(dataloader.dataset)
    print(f'Test Loss: {epoch_loss:.4f}')
    return all_distances, all_labels

test_distances, test_labels = test(model, test_loader, criterion, device)

# Calculate metrics
distances = np.array(test_distances)
labels = np.array(test_labels)
scores = 1 - (distances / distances.max())

auc = roc_auc_score(labels, scores)
fpr, tpr, thresholds = roc_curve(labels, scores)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]
predictions = (scores >= optimal_threshold).astype(float)

print(f'\n=== TEST RESULTS ===')
print(f'AUC-ROC:           {auc:.4f}')
print(f'Optimal Threshold: {optimal_threshold:.4f}')
print(f'Accuracy:          {accuracy_score(labels, predictions):.4f}')
print(f'Precision:         {precision_score(labels, predictions):.4f}')
print(f'Recall:            {recall_score(labels, predictions):.4f}')

# Check target metric
if auc >= 0.80:
    print(f'\n✓ SUCCESS: AUC {auc:.4f} meets epic target of 0.80+')
else:
    print(f'\n⚠ AUC {auc:.4f} is below 0.80 target (opportunity for improvement)')

## Step 9: Visualize evaluation results

The ROC curve summarizes threshold-independent ranking quality on the held-out labeled test split. The distance histogram shows whether related pairs tend to have lower embedding distances than unrelated pairs.


In [ ]:
# Plot ROC curve and distance distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', zorder=5, label=f'Optimal ({optimal_threshold:.2f})')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

pos_dist = distances[labels == 1.0]
neg_dist = distances[labels == 0.0]
axes[1].hist(pos_dist, bins=30, alpha=0.6, label=f'Related (n={len(pos_dist)})')
axes[1].hist(neg_dist, bins=30, alpha=0.6, label=f'Unrelated (n={len(neg_dist)})')
axes[1].set_xlabel('Euclidean Distance')
axes[1].set_ylabel('Count')
axes[1].set_title('Distance Distributions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(work_dir, 'evaluation.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Evaluation plots saved.')

## Summary

- Trained a Siamese network with a pretrained InceptionResnetV1/VGGFace2 face encoder.
- Generated balanced positive/negative kinship pairs from the cleaned relationship data.
- Evaluated on a held-out labeled family split.
- Generated ROC curves and distance-distribution plots.

**Outputs saved to `/kaggle/working/`:**

- `model.pth`: trained model weights.
- `evaluation.png`: ROC curve and distance distributions.
